In [74]:
import pandas as pd
import numpy as np
import requests
from requests.exceptions import HTTPError
import os
from dotenv import load_dotenv
from etl import extract
import datetime

load_dotenv()
access_key = os.getenv('ACCESS_KEY')

# Define API endpoint
url = 'https://www.goflightlabs.com/flights'

# Extracting data from API endpoint
flight_data_raw = extract(url, access_key)
print(flight_data_raw.shape)

Returned status code: 200
(100, 23)


In [78]:
# EDA
display(flight_data_raw.head(), flight_data_raw.info(), flight_data_raw.describe())

print('\nNumber of null values for every column feature\n')
flight_data_raw.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hex            100 non-null    object 
 1   reg_number     100 non-null    object 
 2   flag           100 non-null    object 
 3   lat            100 non-null    float64
 4   lng            100 non-null    float64
 5   alt            100 non-null    int64  
 6   dir            100 non-null    float64
 7   speed          100 non-null    int64  
 8   v_speed        100 non-null    int64  
 9   squawk         4 non-null      object 
 10  flight_number  100 non-null    object 
 11  flight_icao    100 non-null    object 
 12  flight_iata    100 non-null    object 
 13  dep_icao       100 non-null    object 
 14  dep_iata       100 non-null    object 
 15  arr_icao       100 non-null    object 
 16  arr_iata       100 non-null    object 
 17  airline_icao   100 non-null    object 
 18  airline_iat

,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,squawk,...,dep_icao,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type
0,48663E,PH-YHZ,NL,52.107210,6.878214,8813,298.3,702,0,1000,...,LATI,TIA,EHAM,AMS,TRA,HV,A21N,1761337803,en-route,adsb
1,7503D6,9M-LCZ,MY,10.292019,99.995554,10733,163.3,803,0,NaN,...,VGHS,DAC,WMKK,KUL,MXD,OD,B738,1761337803,en-route,adsb
2,A670A3,N514DN,US,45.962216,-104.916183,12714,111.3,924,0,NaN,...,RKSI,ICN,KATL,ATL,DAL,DL,A359,1761337803,en-route,adsb
3,A99E09,N719FR,US,44.925433,-121.191383,6923,110.2,880,0,NaN,...,KDEN,DEN,KATL,ATL,FFT,F9,A21N,1761337803,en-route,adsb
4,AA7E36,N77518,US,21.869235,-104.658772,7251,179.8,740,0,NaN,...,KDEN,DEN,MMPR,PVR,UAL,UA,B738,1761337803,en-route,adsb


None

,lat,lng,alt,dir,speed,v_speed,updated
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.0,1.000000e+02
mean,24.474276,-4.035648,8933.110000,181.211000,763.970000,0.0,1.761338e+09
std,25.497567,84.520059,3406.465688,93.769173,192.215339,0.0,1.000000e-01
min,-39.287039,-156.822063,376.000000,4.500000,4.000000,0.0,1.761338e+09
25%,18.051208,-78.235433,6610.750000,111.825000,699.750000,0.0,1.761338e+09
50%,26.787851,3.186539,10645.500000,178.450000,813.500000,0.0,1.761338e+09
75%,45.211008,71.732478,11403.500000,255.975000,881.000000,0.0,1.761338e+09
max,69.353593,173.780199,13202.000000,358.000000,1049.000000,0.0,1.761338e+09



Number of null values for every column feature



hex               0
reg_number        0
flag              0
lat               0
lng               0
alt               0
dir               0
speed             0
v_speed           0
squawk           96
flight_number     0
flight_icao       0
flight_iata       0
dep_icao          0
dep_iata          0
arr_icao          0
arr_iata          0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
dtype: int64

<h1>Data Cleaning</h1>

<li>Creating a copy of the raw flight data and applying data cleaning</li>
<li>Replacing missing values in "squawk" column with "unknown"</li>
<li>Replacing missing values in 'alt', 'speed' and v_speed to 0</li>
<li>Converting "updated" values from timestamp to datetime</li>


In [79]:
# Copying raw flight data
flight_data_raw_copy = flight_data_raw.copy()

# Replacing missing values with default values
default_vals = {'squawk': 'Unknown', 'alt': 0, 'speed': 0, 'v_speed': 0.0}
flight_data_raw_copy = flight_data_raw.fillna(default_vals)

# Converting 'updated' values from timestamp to datetime
flight_data_raw_copy['updated'] = flight_data_raw_copy['updated'].apply(lambda x: datetime.datetime.fromtimestamp(x))

display(flight_data_raw_copy.head())

print('\nNumber of null values for every column feature\n')
flight_data_raw_copy.isnull().sum()


,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,squawk,...,dep_icao,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type
0,48663E,PH-YHZ,NL,52.107210,6.878214,8813,298.3,702,0,1000,...,LATI,TIA,EHAM,AMS,TRA,HV,A21N,2025-10-24 16:30:03,en-route,adsb
1,7503D6,9M-LCZ,MY,10.292019,99.995554,10733,163.3,803,0,Unknown,...,VGHS,DAC,WMKK,KUL,MXD,OD,B738,2025-10-24 16:30:03,en-route,adsb
2,A670A3,N514DN,US,45.962216,-104.916183,12714,111.3,924,0,Unknown,...,RKSI,ICN,KATL,ATL,DAL,DL,A359,2025-10-24 16:30:03,en-route,adsb
3,A99E09,N719FR,US,44.925433,-121.191383,6923,110.2,880,0,Unknown,...,KDEN,DEN,KATL,ATL,FFT,F9,A21N,2025-10-24 16:30:03,en-route,adsb
4,AA7E36,N77518,US,21.869235,-104.658772,7251,179.8,740,0,Unknown,...,KDEN,DEN,MMPR,PVR,UAL,UA,B738,2025-10-24 16:30:03,en-route,adsb



Number of null values for every column feature



hex              0
reg_number       0
flag             0
lat              0
lng              0
alt              0
dir              0
speed            0
v_speed          0
squawk           0
flight_number    0
flight_icao      0
flight_iata      0
dep_icao         0
dep_iata         0
arr_icao         0
arr_iata         0
airline_icao     0
airline_iata     0
aircraft_icao    0
updated          0
status           0
type             0
dtype: int64